In [1]:
import pandas as pd
import numpy as np
from sklearn import tree, metrics
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_predict

In [2]:
fold_number = 5
number_of_test_samples = 0
dataset = pd.read_csv('HEA42_xrd_0010.csv', index_col=0)

In [3]:
y = dataset.iloc[:, 0]
x = dataset.iloc[:, 1:]

In [4]:
# ランダムにトレーニングデータとテストデータとに分割
if number_of_test_samples == 0:
    x_train = x.copy()
    x_test = x.copy()
    y_train = y.copy()
    y_test = y.copy()
else:
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=number_of_test_samples, shuffle=True, random_state=3, stratify=y)

In [5]:
x # 確認

,N,M,I,T,P
1,0.5,50,5.00,225.0,8.5
2,0.5,50,5.00,265.3,8.1
3,0.5,70,7.00,259.6,8.0
4,0.5,40,4.00,266.0,8.1
5,1.0,70,7.00,261.6,8.1
6,5.0,70,7.00,260.4,8.0
10,5.0,70,9.99,258.8,8.0
11,5.0,70,7.00,269.5,9.5
13,5.0,100,7.00,257.6,8.0
14,5.0,100,7.00,256.3,9.5


In [6]:
param = {'max_depth':[1, 2, 3, 4, 5], 'min_samples_leaf':[1, 2, 3], 'min_samples_split':[2, 3, 4]}

In [7]:
clf = GridSearchCV(tree.DecisionTreeClassifier(), param, cv=10)
clf.fit(x_train, y_train)

GridSearchCV(cv=10, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': [1, 2, 3, 4, 5],
                         'min_samples_leaf': [1, 2, 3],
                         'min_samples_split': [2, 3, 4]})

In [8]:
# スコアとパラメータの組み合わせ
scores = clf.cv_results_['mean_test_score']
params = clf.cv_results_['params']

In [9]:
# 結果の確認
best_clf = clf.best_estimator_
print('best_condition:\n', best_clf)
print('train_score:\n', best_clf.score(x_train, y_train))
print('test_score:\n', best_clf.score(x_test, y_test))
r2cvs = []
for i in range(len(scores)):
    print(scores[i], params[i])
    r2cvs.append(scores[i])

best_condition:
 DecisionTreeClassifier(max_depth=1)
train_score:
 0.9285714285714286
test_score:
 0.9285714285714286
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 3}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 4}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 2, 'min_samples_split': 2}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 2, 'min_samples_split': 3}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 2, 'min_samples_split': 4}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 3, 'min_samples_split': 2}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 3, 'min_samples_split': 3}
0.8550000000000001 {'max_depth': 1, 'min_samples_leaf': 3, 'min_samples_split': 4}
0.8300000000000001 {'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
0.8300000000000001 {'max_depth': 2, 'min_samples_lea

In [10]:
optimal_param = params[np.where(r2cvs==np.max(r2cvs))[0][0]]
optimal_param

{'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2}

In [11]:
optimal_max_depth = optimal_param['max_depth']
optimal_max_depth

1

In [12]:
optimal_min_samples_leaf = optimal_param['min_samples_leaf']
optimal_min_samples_leaf

1

In [13]:
optimal_min_samples_split = optimal_param['min_samples_split']
optimal_min_samples_split

2

In [14]:
# DT 
model = tree.DecisionTreeClassifier(max_depth=optimal_max_depth, min_samples_leaf=optimal_min_samples_leaf)
model.fit(x_train, y_train)

DecisionTreeClassifier(max_depth=1)

In [15]:
# from hkaneko1985/python_data_analysis_ohmsha/sample_functions.py
def estimation_and_performance_check_in_classification_train_and_test(model, x_train, y_train, x_test, y_test):
    class_types = list(set(y_train))  # クラスの種類。これで混同行列における縦と横のクラスの順番を定めます
    class_types.sort(reverse=True)  # 並び替え

    # トレーニングデータのクラスの推定
    estimated_y_train = pd.DataFrame(model.predict(x_train), index=x_train.index, columns=[
        'estimated class'])  # トレーニングデータのクラスを推定し、Pandas の DataFrame 型に変換。行の名前・列の名前も設定

    # トレーニングデータの混同行列
    confusion_matrix_train = pd.DataFrame(
        metrics.confusion_matrix(y_train, estimated_y_train, labels=class_types), index=class_types,
        columns=class_types)  # 混同行列を作成し、Pandas の DataFrame 型に変換。行の名前・列の名前を定めたクラスの名前として設定
    confusion_matrix_train.to_csv('confusion_matrix_train.csv')  # csv ファイルに保存。同じ名前のファイルがあるときは上書きされますので注意してください
    print(confusion_matrix_train)  # 混同行列の表示
    print('Accuracy for training data :', metrics.accuracy_score(y_train, estimated_y_train), '\n')  # 正解率の表示

    # トレーニングデータの結果の保存
    y_train_for_save = pd.DataFrame(y_train)  # Series のため列名は別途変更
    y_train_for_save.columns = ['actual class']
    y_error_train = y_train_for_save.iloc[:, 0] == estimated_y_train.iloc[:, 0]
    y_error_train = pd.DataFrame(y_error_train)  # Series のため列名は別途変更
    y_error_train.columns = ['TRUE if estimated class is correct']
    results_train = pd.concat([estimated_y_train, y_train_for_save, y_error_train], axis=1)
    results_train.to_csv('estimated_y_train.csv')  # 推定値を csv ファイルに保存。同じ名前のファイルがあるときは上書きされますので注意してください

    # テストデータのクラスの推定
    estimated_y_test = pd.DataFrame(model.predict(x_test), index=x_test.index,
                                    columns=['estimated class'])  # テストデータのクラスを推定し、Pandas の DataFrame 型に変換。行の名前・列の名前も設定

    # テストデータの混同行列
    confusion_matrix_test = pd.DataFrame(
        metrics.confusion_matrix(y_test, estimated_y_test, labels=class_types), index=class_types,
        columns=class_types)  # 混同行列を作成し、Pandas の DataFrame 型に変換。行の名前・列の名前を定めたクラスの名前として設定
    confusion_matrix_test.to_csv('confusion_matrix_test.csv')  # csv ファイルに保存。同じ名前のファイルがあるときは上書きされますので注意してください
    print(confusion_matrix_test)  # 混同行列の表示
    print('Accuracy for test data :', metrics.accuracy_score(y_test, estimated_y_test))  # 正解率の表示

    # テストデータの結果の保存
    y_test_for_save = pd.DataFrame(y_test)  # Series のため列名は別途変更
    y_test_for_save.columns = ['actual class']
    y_error_test = y_test_for_save.iloc[:, 0] == estimated_y_test.iloc[:, 0]
    y_error_test = pd.DataFrame(y_error_test)  # Series のため列名は別途変更
    y_error_test.columns = ['TRUE if estimated class is correct']
    results_test = pd.concat([estimated_y_test, y_test_for_save, y_error_test], axis=1)
    results_test.to_csv('estimated_y_test.csv')  # 推定値を csv ファイルに保存。同じ名前のファイルがあるときは上書きされますので注意してください

In [16]:
# トレーニングデータ・テストデータの推定、混同行列の作成、正解率の値の表示、推定値の保存
estimation_and_performance_check_in_classification_train_and_test(model, x_train, y_train, x_test, y_test)

    1   0
1  27   1
0   2  12
Accuracy for training data : 0.9285714285714286 

    1   0
1  27   1
0   2  12
Accuracy for test data : 0.9285714285714286


In [17]:
# 決定木のモデルを確認するための dot ファイルの作成
with open('tree.dot', 'w') as f:
    if model.classes_.dtype == 'object':
        class_names = model.classes_
    else:
        # クラス名が数値のときの対応
        class_names = []
        for class_name_number in range(0, model.classes_.shape[0]):
            class_names.append(str(model.classes_[class_name_number]))
    tree.export_graphviz(model, out_file=f, feature_names=x.columns, class_names=class_names)